# CODE TO WEBSCRAPE FROM KICKSTARTER.com
1) first of all download all the necessary libraries (there are some duplicates but I am too lazy to remove them )

In [ ]:
import json
import re
import html
import ast
import random 
import os 
from pathlib import Path
from bs4 import BeautifulSoup
import pandas as pd 
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait 
from selenium.webdriver.support import expected_conditions as EC
from selenium.webdriver.chrome.service import Service
from webdriver_manager.chrome import ChromeDriverManager
from bs4 import BeautifulSoup
from selenium.webdriver.chrome.service import Service
from webdriver_manager.chrome import ChromeDriverManager
import time 
from bs4 import BeautifulSoup
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from webdriver_manager.chrome import ChromeDriverManager




# INSTRUCTIONS: 
- scrapde_urls.txt keeps track of every URL scraped (doesn't matter if valid description or not), so that we don't include the same one twice 
- kickstarter_scraped_all.csv accumulates only the valid projects


# Kickstarter dataset 


### First try 
I first try to use a dataset from kaggle that includes a lot of projects from kickstarter. The problem as you can clearly see here is that the description of the project is not available, the only thing available is the 'blurb' which I assume is like a very short one-line description of the project. I think it doesn't really make sense to use those for our project since as you can see in the code below, the mean length of those strings is about 18-19 letters, which seems too low for the NLP analysis we want to do

Using the Blurb is probably not enough 

### SECOND TRY 
Since the dataset doesn't work, I tried a different thing. I first tried to make a loop that basically went to the kickstarter.com website, looked at the popular page, scraped the descriptions of the projects that are on that first page, then goes to page 2 and does the same, then to page 3 and so on until it reaches n amount of descriptions saved. The problem with that was that I wasn't able to construct a loop that switched to the second page under popular and selected those projects, the loop kept only showing the first page and so only loading up to like 12 projects. 

### FINAL IDEA 

Since the above approach didn't work, I found a website that takes monthly screenshots of the whole website, the page is 'https://webrobots.io/kickstarter-datasets/', there you can find and download the zip file that contains the screenshot of the webiste for the month. In that zip there are a bunch of CSVs that need to be put together (they are different file because putting them all in one single CSV would be too large). So we first get those and put them all together and from that dataset we only want the links (these are the links to the official kickstarter page for each project), then we use these links in a loop that extracts the descriptions by webscraping and also gets the % of how much each project got funded. 


In this case just to test the code I first selected randomly 100 urls from the dataset and then webscraped the information for those

In [ ]:
import pandas as pd
import glob


folder_path = "Kickstarter_2026-03-12T03_20_26_556Z"

files = glob.glob(f"{folder_path}/*.csv")

print("Number of files found:", len(files))

dfs = []

for file in files:
    print("Loading:", file)
    df_temp = pd.read_csv(file)
    dfs.append(df_temp)

df = pd.concat(dfs, ignore_index=True)

print("Final shape:", df.shape)

In [ ]:

category_names = []

for item in df['category']:
    parsed = json.loads(item)
    name = parsed['name']
    if name not in category_names:
        category_names.append(name)

print(category_names, len(category_names))


parent_names = set()

for item in df['category']:
    parsed = ast.literal_eval(item)
    if 'parent_name' in parsed:
        parent_names.add(parsed['parent_name'])

print(parent_names)
print(len(parent_names))

category_counts = {}

for item in df['category']:
    parsed = ast.literal_eval(item)
    if 'parent_name' in parsed:
        parent = parsed['parent_name']
        if parent in category_counts:
            category_counts[parent] += 1
        else:
            category_counts[parent] = 1

print(category_counts)



12k projects => 10 categories 
15k => 7 categories 
20k => 6 categories 
22k => 5 categories 

In [ ]:
final_categories = {}
for item in category_counts:
    if category_counts[item] > 22000:
        final_categories[item] = category_counts[item]

final_categories

# POLISHING THE DATASET 

In [ ]:
copy = df[df['country'] == 'US']
copy = copy[copy['usd_pledged'] > 0]
copy['pct_goal_reached'] = (copy['usd_pledged'] / copy['goal']) * 100
copy = copy[copy['pct_goal_reached'] > 1]
copy = copy[copy['usd_pledged'] == copy['converted_pledged_amount']]
copy = copy[copy['state'] != 'canceled']

copy = copy[copy['goal'] > 5000]


def get_parent_name(item):
    parsed = ast.literal_eval(item)
    return parsed.get('parent_name', None)

copy['parent_name'] = copy['category'].apply(get_parent_name)
copy = copy[copy['parent_name'].isin(final_categories.keys())]

len(copy)

In [ ]:
copy.iloc[4,1]

In [ ]:
def extract_project_url(x):
    if pd.isna(x):
        return None
    try:
        d = json.loads(x)
        return d.get("web", {}).get("project")
    except:
        return None
    
copy["project_url"] = copy['urls'].apply(extract_project_url)

print(copy["project_url"].notna().sum())
print(copy["project_url"].dropna().head())

# here is the code to actually run the loop for all the urls you extracted in project_urls: 


## NOT SO RANDOM Sampling: 


In [ ]:


df_valid = copy[copy['project_url'].notna()]
# If a tracking file exists, load already-scraped URLs
if os.path.exists('scraped_urls.txt'):
    with open('scraped_urls.txt', 'r') as f:
        scraped_urls = set(f.read().splitlines())
else:
    scraped_urls = set()

# Filter out already-scraped URLs
df_remaining = df_valid[~df_valid["project_url"].isin(scraped_urls)]
print(f"Already scraped: {len(scraped_urls)}")
print(f"Remaining: {len(df_remaining)}")
print(df_remaining['parent_name'].value_counts())

In [ ]:
len(df_remaining)

In [ ]:
n_categories = df_remaining['parent_name'].nunique()
batch_size = 100

n_per_category = batch_size // n_categories

sample_balanced = (
    df_remaining
    .groupby('parent_name')
    .sample(n=n_per_category)
)

batch_urls = sample_balanced["project_url"].tolist()
print(f"Categories: {n_categories}, per category: {n_per_category}, total: {len(batch_urls)}")
print(sample_balanced['parent_name'].value_counts())
print(len(batch_urls), 'projects :', batch_urls)

In [ ]:

def scrape_kickstarter_description(url: str) -> dict:
    options = webdriver.ChromeOptions()
    options.add_argument("--headless")
    options.add_argument("--no-sandbox")
    options.add_argument("--disable-dev-shm-usage")
    options.add_argument("--disable-gpu") 
    options.add_argument("--window-size=1920,1080")       
    options.add_argument("--disable-extensions")          
    options.add_argument("--disable-software-rasterizer")  
    options.add_argument("--remote-debugging-port=0")     
    options.add_argument(
        "user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 (KHTML, like Gecko) "
        "Chrome/120.0.0.0 Safari/537.36"
    )

    driver = webdriver.Chrome(
        service=Service(ChromeDriverManager().install()),
        options=options
    )

    try:
        driver.get(url)

        WebDriverWait(driver, 15).until(
            EC.presence_of_element_located((By.CLASS_NAME, "rte__content"))
        )

        time.sleep(2)
        html_source = driver.page_source
        soup = BeautifulSoup(html_source, "html.parser")

        
        description_div = soup.find("div", class_="rte__content")
        description = (
            description_div.get_text(separator=" ", strip=True)
            if description_div else None
        )

        
        title = soup.find("h1", class_="project-name")
        if not title:
            title = soup.find("h2")
        title = title.get_text(strip=True) if title else None

        funding = {
            "pledged": None,
            "usd_pledged": None,
            "converted_pledged_amount": None,
            "goal": None,
            "currency": None
        }

        for script in soup.find_all("script"):
            script_text = script.string or script.get_text()
            if "window.current_project" not in script_text:
                continue

            match = re.search(
                r'window\.current_project\s*=\s*"(.+?)";',
                script_text,
                re.DOTALL
            )
            if not match:
                continue

            raw = match.group(1)

            raw = html.unescape(raw)
            try:
                raw = raw.encode("utf-8").decode("unicode_escape")
            except Exception: 
                pass

            try:
                project_data = json.loads(raw)
                funding = {
                    "pledged": project_data.get("pledged"),
                    "usd_pledged": project_data.get("usd_pledged"),
                    "converted_pledged_amount": project_data.get("converted_pledged_amount"),
                    "goal": project_data.get("goal"),
                    "currency": project_data.get("currency")
                }
                break
            except json.JSONDecodeError:
                pass

    finally:
        driver.quit()

    return {
        "url": url,
        "title": title,
        "description": description,
        **funding
    }





In [ ]:
MIN_DESCRIPTION_LENGTH = 800
output_file = 'kickstarter_scraped_all.csv'
rows = []

for i, url in enumerate(batch_urls, start=1):
    print(f"Scraping {i}/{len(batch_urls)}: {url}")

    try:
        row = scrape_kickstarter_description(url)
    except Exception as e:
        print(f"  ✗ Error scraping {url}: {e}")
        continue


    desc = row.get("description")
    if desc and len(desc) >= MIN_DESCRIPTION_LENGTH:
        rows.append(row)
        scraped_urls.add(url)
        print(f"  ✓ Saved (length: {len(desc)})")
        time.sleep(2)
    else:
        print(f"  ⚠ Skipped (description too short or missing)")

    # save progress every 10 URLs
    if i % 10 == 0:
        if rows:
            pd.DataFrame(rows).to_csv(output_file, mode='a', header=not os.path.exists(output_file), index=False)
            rows = []  # clear rows after saving to avoid duplicates
        with open('scraped_urls.txt', 'w') as f:
            f.write('\n'.join(scraped_urls))
        print(f"  💾 Progress saved at {i}/{len(batch_urls)}")

    

if rows:
    pd.DataFrame(rows).to_csv(output_file, mode='a', header=not os.path.exists(output_file), index=False)

with open('scraped_urls.txt', 'w') as f:
    f.write('\n'.join(scraped_urls))

print(f"\nBatch complete.")
print(f"Total URLs scraped so far: {len(scraped_urls)}")

In [ ]:
file = pd.read_csv('kickstarter_scraped_all.csv')

In [ ]:
file[file['description'] != '']

saving them in a dataset:

Possible NLP Tasks: 
- classification: classify based on description if campaing reaches x% of funding or 100% of funding (TF-IDF and logistic regression or BERT CLassifier) 
- Topic modelling: identify the main topics from each campaign and do what? 
- Embedding analysis: either transform all the descriptions into vectors with doc2vec and do clusters based on cosine distance (maybe also compare which cluster has higher funding)
- Generative Task: take a real campaign and ask to rewrite it in a more persuasive tone or just better, then feed into the initial classifier and test if gets better classification 

NOTEs

- thing that she would like to see is a preliminary analysis on the data
- don't take two weeks to download the data 
